# Problema 3 — Clusterização: "O Perfil do Consumidor"

**Integrante:** Gean Carlos

## Objetivo

Agrupar clientes de um shopping center por similaridade, com base em renda anual e pontuação de gastos (*Spending Score*), usando aprendizado não supervisionado. Como não existe rótulo pré-definido para cada cliente, o algoritmo **K-Means** identifica os grupos e a interpretação comercial de cada um é feita por nós.

O trabalho inclui o escalonamento das variáveis, a escolha do número de clusters pelo Método do Cotovelo, o treinamento do modelo com visualização dos grupos e a atribuição de nomes comerciais a cada cluster.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Reprodutibilidade dos experimentos
RANDOM_STATE = 42

# Exibição de gráficos dentro do notebook
%matplotlib inline

sns.set_theme(style="whitegrid")

print("Bibliotecas importadas com sucesso!")

In [ ]:
try:
    # No Google Colab: abre a caixa de upload do arquivo CSV
    from google.colab import files
    uploaded = files.upload()
    arquivo = next(
        nome for nome in uploaded
        if nome.lower().endswith(".csv")
    )
except ImportError:
    # Fora do Colab: lê o arquivo da pasta atual
    arquivo = "Mall_Customers.csv"

df = pd.read_csv(arquivo)

print("Dimensões do dataset:", df.shape)

display(df.head())

print("\nTipos das colunas:")
display(df.dtypes.to_frame("tipo"))

print("\nValores ausentes por coluna:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("valores_ausentes")
)

print("\nEstatísticas descritivas:")
display(df.describe().round(2))

In [ ]:
# Nomes das colunas usadas na clusterização
COL_RENDA = "Annual Income (k$)"
COL_GASTO = "Spending Score (1-100)"

# Conferir se as colunas existem e se há duplicados
print("Colunas encontradas:", list(df.columns))
print("Linhas completamente duplicadas:", df.duplicated().sum())

# Criar uma cópia para preservar o dataset original
df_limpo = df.dropna(subset=[COL_RENDA, COL_GASTO]).drop_duplicates().copy()

print("Registros após a limpeza:", len(df_limpo))

### Questão 1 — Escalonamento

O K-Means é um algoritmo baseado em **distância euclidiana**: ele agrupa os pontos que estão mais próximos uns dos outros. Quando as variáveis têm grandezas muito diferentes, como uma renda na casa das dezenas ou centenas de milhares e uma idade na casa das dezenas, a variável de maior escala domina o cálculo da distância e a outra praticamente deixa de influenciar o resultado. O modelo passaria a agrupar quase só pela renda, distorcendo os clusters.

Para resolver esse problema, foi aplicada a **padronização (z-score)** com o `StandardScaler`, que transforma cada variável usando a fórmula:

$$z = \frac{x - \mu}{\sigma}$$

em que $\mu$ é a média e $\sigma$ é o desvio padrão da variável. Depois da padronização, cada variável passa a ter **média 0 e desvio padrão 1**, e todas contribuem com o mesmo peso para o cálculo das distâncias.

Neste conjunto de dados, a renda está em milhares de dólares (de 15 a 137) e o *Spending Score* varia de 1 a 99, com desvios padrão muito parecidos (cerca de 26 em ambas). Por isso, o efeito prático da padronização aqui é pequeno. Ainda assim, ela é uma boa prática e se torna indispensável quando as escalas são muito diferentes, como no exemplo do enunciado (renda de 100.000 contra idade de 30) ou se a variável `Age` fosse incluída no modelo.

In [ ]:
# Variáveis usadas no treinamento (conforme pedido no enunciado)
X = df_limpo[[COL_RENDA, COL_GASTO]].copy()

# Padronização: média 0 e desvio padrão 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Conferir o resultado do escalonamento
comparacao = pd.DataFrame({
    "variavel": [COL_RENDA, COL_GASTO],
    "media_antes": X.mean().values,
    "desvio_antes": X.std().values,
    "media_depois": X_scaled.mean(axis=0),
    "desvio_depois": X_scaled.std(axis=0),
}).round(3)

print("Antes e depois do escalonamento:")
display(comparacao)

In [ ]:
# Método do Cotovelo: inércia para K de 1 a 10
valores_k = range(1, 11)
inercias = []
silhuetas = {}

for k in valores_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

    # O silhouette score só é definido para K >= 2
    if k >= 2:
        silhuetas[k] = silhouette_score(X_scaled, km.labels_)

# Gráfico do cotovelo
plt.figure(figsize=(9, 5))
plt.plot(list(valores_k), inercias, marker="o")
plt.xticks(list(valores_k))
plt.title("Método do Cotovelo")
plt.xlabel("Número de clusters (K)")
plt.ylabel("Inércia (soma das distâncias ao centróide)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Silhouette como apoio à decisão
print("Silhouette score por K:")
display(pd.Series(silhuetas, name="silhouette").round(3).to_frame())

### Questão 2 — O Dilema da Quantidade

No aprendizado não supervisionado não existe um número de grupos conhecido de antemão. Para escolher o valor de K, foi aplicado o **Método do Cotovelo**: o K-Means foi treinado com K de 1 a 10 e, para cada valor, foi calculada a **inércia**, que é a soma das distâncias entre cada ponto e o centróide do seu cluster.

A inércia sempre diminui quando K aumenta, porque os grupos ficam menores e mais próximos dos seus centróides. O ponto de interesse é o "cotovelo" do gráfico, onde a queda deixa de ser acentuada e passa a ser pequena. A partir desse ponto, criar mais clusters aumenta a complexidade do modelo sem trazer uma melhora relevante.

No gráfico, a curva cai de forma acentuada até **K = 5** (a inércia ainda diminui cerca de 40% ao passar de K=4 para K=5) e depois passa a diminuir pouco (cerca de 16% de K=5 para K=6, e menos que isso nos valores seguintes). Por isso, foram escolhidos **5 clusters**.

O *silhouette score*, que mede o quão bem separados estão os grupos, foi usado como apoio: ele atinge o **valor máximo em K = 5 (0,555)**, acima dos valores de K=4 (0,494) e K=6 (0,540), confirmando a escolha feita pelo cotovelo.

Além disso, cinco grupos são um número gerenciável para o time de marketing criar campanhas personalizadas, sem exigir uma quantidade excessiva de ações diferentes.

In [ ]:
# Número de clusters escolhido na análise do cotovelo
K = 5

modelo_kmeans = KMeans(
    n_clusters=K,
    random_state=RANDOM_STATE,
    n_init=10
)

# Treinar e atribuir um cluster a cada cliente
df_limpo["cluster"] = modelo_kmeans.fit_predict(X_scaled)

print("Quantidade de clientes em cada cluster:")
display(
    df_limpo["cluster"]
    .value_counts()
    .sort_index()
    .to_frame("quantidade_clientes")
)

# Centróides de volta para a escala original (para plotar e interpretar)
centroides = pd.DataFrame(
    scaler.inverse_transform(modelo_kmeans.cluster_centers_),
    columns=[COL_RENDA, COL_GASTO]
).round(1)

print("\nCentróides (escala original):")
display(centroides)

In [ ]:
# Gráfico de dispersão colorido por cluster
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=df_limpo,
    x=COL_RENDA,
    y=COL_GASTO,
    hue="cluster",
    palette="viridis",
    s=70,
    edgecolor="black",
    linewidth=0.3
)

# Centróides
plt.scatter(
    centroides[COL_RENDA],
    centroides[COL_GASTO],
    c="red",
    marker="X",
    s=250,
    edgecolor="black",
    label="Centróides"
)

for i, linha in centroides.iterrows():
    plt.annotate(
        str(i),
        (linha[COL_RENDA], linha[COL_GASTO]),
        textcoords="offset points",
        xytext=(10, 10),
        fontweight="bold"
    )

plt.title(f"Segmentação de clientes com K-Means (K = {K})")
plt.xlabel("Renda Anual (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Questão 3 — Treinamento e Visualização

O modelo K-Means foi treinado com **5 clusters**, usando apenas as variáveis `Annual Income (k$)` e `Spending Score (1-100)`, já padronizadas. As demais colunas do conjunto de dados não participaram do treinamento.

No gráfico de dispersão, cada ponto representa um cliente, colorido de acordo com o cluster ao qual foi atribuído, e os centróides estão marcados com um **X vermelho**. Os grupos aparecem bem definidos nos eixos de renda e gastos, o que indica que o algoritmo conseguiu separar perfis de consumo distintos.

Os 200 clientes ficaram distribuídos em grupos de tamanhos diferentes: 81, 39, 22, 35 e 23 clientes (para a semente `random_state=42`). A tabela com a quantidade de clientes por cluster mostra o tamanho de cada segmento, informação importante para priorizar campanhas de marketing.

In [ ]:
# Perfil médio de cada cluster
colunas_perfil = [COL_RENDA, COL_GASTO]
if "Age" in df_limpo.columns:
    colunas_perfil.append("Age")

perfil = (
    df_limpo
    .groupby("cluster")[colunas_perfil]
    .mean()
    .round(1)
)
perfil["clientes"] = df_limpo["cluster"].value_counts().sort_index()

print("Perfil médio de cada cluster:")
display(perfil)

# Nomeação dos clusters a partir dos centróides, independente do número do cluster.
# Cada cluster é comparado com a média geral (em desvios padrão).
def nomear_clusters(perfil):
    z_renda = (perfil[COL_RENDA] - df_limpo[COL_RENDA].mean()) / df_limpo[COL_RENDA].std()
    z_gasto = (perfil[COL_GASTO] - df_limpo[COL_GASTO].mean()) / df_limpo[COL_GASTO].std()

    # O cluster "do meio" é o de centróide mais próximo da média geral
    moderado = (z_renda**2 + z_gasto**2).idxmin()

    nomes = {}
    for c in perfil.index:
        if c == moderado:
            nomes[c] = "Os Moderados"
        elif z_renda[c] > 0 and z_gasto[c] > 0:
            nomes[c] = "Os VIPs"
        elif z_renda[c] > 0:
            nomes[c] = "Os Econômicos"
        elif z_gasto[c] > 0:
            nomes[c] = "Os Impulsivos"
        else:
            nomes[c] = "Os Contidos"
    return nomes

nomes_clusters = nomear_clusters(perfil)
perfil["perfil_comercial"] = perfil.index.map(nomes_clusters)

print("\nNome comercial atribuído a cada cluster:")
display(perfil[[COL_RENDA, COL_GASTO, "clientes", "perfil_comercial"]])

In [ ]:
# Aplicar os nomes comerciais a cada cliente
df_limpo["perfil_comercial"] = df_limpo["cluster"].map(nomes_clusters)

# Gráfico final com os nomes comerciais
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=df_limpo,
    x=COL_RENDA,
    y=COL_GASTO,
    hue="perfil_comercial",
    palette="Set1",
    s=70,
    edgecolor="black",
    linewidth=0.3
)

plt.title("Perfis de consumidores do shopping")
plt.xlabel("Renda Anual (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Perfil", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display(
    df_limpo
    .groupby("perfil_comercial")[[COL_RENDA, COL_GASTO]]
    .agg(["mean", "count"])
    .round(1)
)

### Questão 4 — Interpretação de Negócios (O Rótulo Humano)

O K-Means apenas agrupa os clientes por similaridade. O significado de cada grupo é definido por nós a partir da posição dos clusters no gráfico, considerando renda e gastos. Os nomes comerciais atribuídos foram:

| Perfil | Renda | Gastos | Descrição | Ação de marketing sugerida |
|---|---|---|---|---|
| **Os VIPs** | Alta | Alto | Clientes com muito poder de compra que também consomem bastante. | Programa de fidelidade, atendimento exclusivo e lançamentos antecipados. |
| **Os Econômicos** | Alta | Baixo | Têm dinheiro, mas gastam pouco no shopping. | Ofertas exclusivas e experiências premium para despertar interesse. |
| **Os Moderados** | Média | Médio | Perfil equilibrado, sem extremos de renda ou gastos. | Promoções sazonais e combos que incentivem a recorrência. |
| **Os Impulsivos** | Baixa | Alto | Gastam acima do que a renda sugere. | Parcelamento, cupons e campanhas de oportunidade relâmpago. |
| **Os Contidos** | Baixa | Baixo | Consumo controlado, sensíveis a preço. | Liquidações, descontos progressivos e cupons de baixo valor. |

Os valores médios de cada grupo, obtidos na execução, confirmam esses perfis: VIPs (renda ≈ 86,5 k$ e gasto ≈ 82), Econômicos (renda ≈ 88,2 k$ e gasto ≈ 17), Moderados (renda ≈ 55,3 k$ e gasto ≈ 50), Impulsivos (renda ≈ 25,7 k$ e gasto ≈ 79) e Contidos (renda ≈ 26,3 k$ e gasto ≈ 21).

Embora a idade não tenha sido usada no treinamento, ela ajuda a caracterizar os grupos: os Impulsivos são os mais jovens (média de cerca de 25 anos), os VIPs também são relativamente jovens (cerca de 33 anos) e os Contidos são os mais velhos (cerca de 45 anos). Os nomes são atribuídos automaticamente pelos centróides de cada cluster, então a correspondência com o número do cluster não depende da execução. Com a segmentação, o time de marketing consegue criar campanhas personalizadas para cada grupo, em vez de uma comunicação única para todos os clientes.

### Conclusão

A padronização das variáveis foi essencial para que renda e gastos tivessem o mesmo peso no K-Means. O Método do Cotovelo indicou 5 clusters, e o modelo separou perfis de consumo distintos, que receberam nomes comerciais e ações de marketing específicas. Como limitação, o K-Means assume grupos de formato aproximadamente esférico e depende da escolha de K, por isso a interpretação humana dos clusters continua sendo fundamental.